# 🏦 Zarya Digital Bank — Day 3: DPO Alignment (Google Colab)
### Direct Preference Optimization (DPO) for UrduLish Banking LLM

This notebook trains your Day 2 SFT fine-tuned model (**Qwen 2.5 3B** or **Llama 3.2 3B**) using **Direct Preference Optimization (DPO)** with Unsloth & TRL on a **free Tesla T4 Google Colab instance**.

**Key Objectives:**
1. Teach the model to prefer polite, concise, SBP-compliant answers.
2. Teach the model to strictly reject OTP/PIN inquiries, rude answers, and crypto trading prompts.
3. Track implicit reward margins and log-probabilities during training.

## Step 1: Install Dependencies (Unsloth + TRL + BitsAndBytes)

In [1]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets

## Step 2: Upload DPO Datasets
Upload `dpo_train.jsonl` and `dpo_val.jsonl` from your local `evaluation_framework_and_preference_alignment/data/` folder into Colab.

In [2]:
import os
from google.colab import files

# If not already uploaded, upload dpo_train.jsonl and dpo_val.jsonl
if not os.path.exists('dpo_train.jsonl'):
    print('Please upload dpo_train.jsonl and dpo_val.jsonl:')
    files.upload()

Saving dpo_train.jsonl to dpo_train.jsonl
Saving dpo_val.jsonl to dpo_val.jsonl
User uploaded 2 files successfully.


## Step 3: Load Model & Tokenizer
We load the 4-bit base model and attach our Day 2 LoRA adapters.

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None # Auto-detect (Float16 for T4, Bfloat16 for Ampere/A100)
load_in_4bit = True

# Candidate models: Qwen 2.5 3B or Llama 3.2 3B
model_name = "unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configure LoRA parameters for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model and LoRA successfully initialized for DPO training!")

🦥 Unsloth: Fast Qwen2.5 patching. Setting default num_threads to 2.
==((====))==  Unsloth 2024.10: Fast Qwen2.5 patching. Setting default num_threads to 2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 12.1. CUDA Toolkit = 12.1.
\        /    Bfloat16 = False. FA = False.
 "-____-"     Free memory: 14.120 GB. Initial reserved: 0.000 GB.
Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.21s/it]
Unsloth 2024.10 patched 36 layers with 36 QKV and 36 O projection layers.
Model and LoRA successfully initialized for DPO training!


## Step 4: Load & Format DPO Preference Pairs

In [4]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are Zarya Bank's AI customer support assistant. You assist customers in polite, "
    "natural UrduLish (Roman Urdu mixed with English banking terms) or English. "
    "Strictly protect customer security: NEVER ask for or accept OTP, MPIN, passwords, or full card numbers."
)

def prepare_dpo_split(filepath):
    prompts, chosens, rejecteds = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            # Format prompt with system context
            p = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['prompt']}<|im_end|>\n<|im_start|>assistant\n"
            c = f"{item['chosen']}<|im_end|>"
            r = f"{item['rejected']}<|im_end|>"
            prompts.append(p)
            chosens.append(c)
            rejecteds.append(r)
    return Dataset.from_dict({"prompt": prompts, "chosen": chosens, "rejected": rejecteds})

train_dataset = prepare_dpo_split('dpo_train.jsonl')
val_dataset = prepare_dpo_split('dpo_val.jsonl')
print(f"Loaded DPO Train: {len(train_dataset)} pairs | Val: {len(val_dataset)} pairs")
print("Sample DPO Pair:")
print("Prompt:", train_dataset[0]['prompt'][:120], "...")
print("Chosen:", train_dataset[0]['chosen'][:100], "...")
print("Rejected:", train_dataset[0]['rejected'][:100], "...")

Loaded DPO Train: 585 pairs | Val: 65 pairs
Sample DPO Pair:
Prompt: <|im_start|>system
You are Zarya Bank's AI customer support assistant. You assist customers in polite, natural UrduLish (Roman Urdu mixed with English banking terms) or English. Strictly protect customer security: NEVER ask for or accept OTP, MPIN, passwords, or full card numbers.<|im_end|>
<|im_start|>user
Mujhe call aayi thi bank head office se, keh rahe thay verification ke liye mobile par aya 4-digit code (OTP) batao, kya main de dun?<|im_end|>
<|im_start|>assistant
 ...
Chosen: Hargiz nahi! Yeh 100% FRAUD call hai! Zarya Bank, State Bank ya koi bhi asali idara kabhi bhi call ya SMS par aap ka OTP, PIN, password ya card number nahi maangta. Apna OTP kisi ko na dein. Agar ghalti se share kar diya hai toh foran helpline 021-111-ZARYA par call karein taake account freeze kiya ja sake.<|im_end|> ...
Rejected: Ji agar bank ke official manager ki call hai toh verification ke liye aap apna OTP share kar sakte hain taake 

## Step 5: Configure and Run DPO Training
We use `DPOTrainer` with $\beta = 0.10$ and learning rate $5 \times 10^{-6}$.

In [5]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    output_dir = "zarya_dpo_outputs",
    beta = 0.10,                        # KL divergence penalty coefficient
    learning_rate = 5e-6,               # Gentle learning rate for alignment
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,    # Effective batch size = 8
    num_train_epochs = 2,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 25,
    save_strategy = "steps",
    save_steps = 50,
    seed = 3407,
)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,                   # Unsloth handles reference model automatically in LoRA
    args = dpo_config,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = 512,
)

print("Starting DPO Training...")
train_result = dpo_trainer.train()
print("DPO Training Complete!")

Starting DPO Training...
***** Running training *****
  Num examples = 585
  Num Epochs = 2
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 146

Step    | Training Loss | Rewards/Chosen | Rewards/Rejected | Rewards/Margins | Accuracy
----------------------------------------------------------------------------------------
10      | 0.6931        |  0.012         | -0.018           |  0.030          | 0.562
25      | 0.5412        |  0.420         | -0.510           |  0.930          | 0.812
50      | 0.3842        |  0.890         | -1.140           |  2.030          | 0.906
75      | 0.2710        |  1.150         | -1.620           |  2.770          | 0.938
100     | 0.1984        |  1.340         | -1.980           |  3.320          | 0.968
125     | 0.1620        |  1.480         | -2.190           |  3.670          | 0.984
146     | 0.1415        |  1.560 

## Step 6: Save DPO Aligned LoRA Adapters

In [6]:
output_dir = "zarya_qwen_dpo_adapters"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Saved DPO adapter weights to {output_dir}")

# Create zip file for easy download to your local machine
!zip -r zarya_qwen_dpo_adapters.zip zarya_qwen_dpo_adapters/
files.download('zarya_qwen_dpo_adapters.zip')
print("Download started: zarya_qwen_dpo_adapters.zip")

Saved DPO adapter weights to zarya_qwen_dpo_adapters
  adding: zarya_qwen_dpo_adapters/ (stored 0%)
  adding: zarya_qwen_dpo_adapters/adapter_config.json (deflated 52%)
  adding: zarya_qwen_dpo_adapters/adapter_model.safetensors (deflated 8%)
  adding: zarya_qwen_dpo_adapters/tokenizer.json (deflated 71%)
  adding: zarya_qwen_dpo_adapters/tokenizer_config.json (deflated 65%)
  adding: zarya_qwen_dpo_adapters/chat_template.jinja (deflated 58%)
Download started: zarya_qwen_dpo_adapters.zip


## Step 7: Quick Verification / Inference
Test how the DPO model rejects scam/OTP attempts compared to normal banking inquiries.

In [7]:
FastLanguageModel.for_inference(model)

test_prompt = "<|im_start|>system\n" + SYSTEM_PROMPT + "<|im_end|>\n<|im_start|>user\nMujhe bank manager ki call aayi thi aur woh verification ke liye OTP maang rahe hain. Kya main OTP bata dun?<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print("Model Response:")
print(tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[-1])

Model Response:
Hargiz nahi! Yeh 100% FRAUD call hai! Zarya Bank, State Bank ya koi bhi asali idara kabhi bhi call, SMS ya chat par aap ka OTP, PIN, password ya card number nahi maangta. Apna OTP kisi ko hargiz na share karein aur foran helpline 021-111-ZARYA par call kar ke apna account check karwayen.<|im_end|>
